In [1]:
# uv 프로젝트 환경이 있으면 uv sync로 의존성 설치 (권장)
!pip install numpy pandas matplotlib seaborn scipy scikit-learn statsmodels pingouin scikit_posthocs xgboost unidecode geopandas -q
print("pip 설치 완료!")

pip 설치 완료!



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\82105\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
from unidecode import unidecode
import geopandas as gpd

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝
from sklearn.preprocessing import MinMaxScaler

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)



라이브러리 로드 완료!
한글 폰트 설정 완료!


In [4]:
df_geolocation = pd.read_csv('../data/olist_geolocation_dataset.csv')
print('데이터 로드 완료')

데이터 로드 완료


In [5]:
dup_zip = df_geolocation['geolocation_zip_code_prefix'].duplicated(keep = False) #우편번호가 중복인 데이터
df_dup = df_geolocation[dup_zip][['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng']].sort_values('geolocation_zip_code_prefix')
#우편번호가 중복인 해당 컬럼 3개

#df_dup.head()

In [ ]:
#중복된 우편번호에 따른 경도와 위도 그래프 시각화

# df_range = df_dup.groupby('geolocation_zip_code_prefix').agg(
#     lat_range=('geolocation_lat', lambda x: x.max() - x.min()),
#     lng_range=('geolocation_lng', lambda x: x.max() - x.min())
# ).reset_index()

# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# axes[0].hist(df_range['lat_range'], bins=50)
# axes[0].set_title('우편번호별 위도 차이 분포')
# axes[0].set_xlabel('위도 범위 (degree)')
# axes[0].set_ylabel('우편번호 개수')
# axes[0].set_xlim(0, 5)

# axes[1].hist(df_range['lng_range'], bins=50)
# axes[1].set_title('우편번호별 경도 차이 분포')
# axes[1].set_xlabel('경도 범위 (degree)')
# axes[1].set_ylabel('우편번호 개수')
# axes[1].set_xlim(0, 5)

# plt.tight_layout()
# plt.show()

In [ ]:
# #산점도 시각화

# plt.scatter(df_range['lat_range'], df_range['lng_range'], alpha=0.3)
# plt.xlabel('위도 차이 (degree)')
# plt.ylabel('경도 차이 (degree)')
# plt.title('우편번호별 위도/경도 차이')
# plt.show()

In [6]:
#브라질 위도와 경도에 따른 필터링

#위도(lat) -34~6, 경도(lng) -74.5~-32

df_geolocation = df_geolocation[
    (df_geolocation['geolocation_lng'] < -32) &
    (df_geolocation['geolocation_lng'] > -74.5) &
    (df_geolocation['geolocation_lat'] < 6) &
    (df_geolocation['geolocation_lat'] > -34)
]

In [ ]:
# #시각화

# # URL에서 직접 브라질 지도 불러오기
# world = gpd.read_file("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip")
# brazil = world[world['NAME'] == 'Brazil']


# fig, ax = plt.subplots(figsize=(12, 10))

# brazil.plot(ax=ax, color='lightgrey', edgecolor='black')

# ax.scatter(df_geolocation['geolocation_lng'], df_geolocation['geolocation_lat'],
#            s=1, alpha=0.3, color='blue')

# ax.set_xlim(-80, -20) #경도 비율 조절
# ax.set_ylim(-40, 20) #위도 비율 조절

# ax.set_title('브라질 우편번호 좌표 분포')
# ax.set_xlabel('경도')
# ax.set_ylabel('위도')
# plt.show()

In [7]:
#위도와 경도의 차를 통하여 필터링

print(df_geolocation.shape)

df_geolocation = df_geolocation.groupby('geolocation_zip_code_prefix').filter(
    lambda x: (x['geolocation_lat'].max() - x['geolocation_lat'].min() < 1.5) &
              (x['geolocation_lng'].max() - x['geolocation_lng'].min() < 1.5)
)

print(df_geolocation.shape)

(1000132, 5)
(988050, 5)


In [ ]:
# #필터링 후 시각화

# # 필터링 후 df_dup 다시 계산
# dup_zip2 = df_geolocation['geolocation_zip_code_prefix'].duplicated(keep=False)
# df_dup2 = df_geolocation[dup_zip2][['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng']]

# df_range2 = df_dup2.groupby('geolocation_zip_code_prefix').agg(
#     lat_range=('geolocation_lat', lambda x: x.max() - x.min()),
#     lng_range=('geolocation_lng', lambda x: x.max() - x.min())
# ).reset_index()

# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# axes[0].hist(df_range2['lat_range'], bins=50)
# axes[0].set_title('우편번호별 위도 차이 분포')
# axes[0].set_xlabel('위도 범위 (degree)')
# axes[0].set_ylabel('우편번호 개수')
# axes[0].set_xlim(0, 1.5)

# axes[1].hist(df_range2['lng_range'], bins=50)
# axes[1].set_title('우편번호별 경도 차이 분포')
# axes[1].set_xlabel('경도 범위 (degree)')
# axes[1].set_ylabel('우편번호 개수')
# axes[1].set_xlim(0, 1.5)

# plt.tight_layout()
# plt.show()

In [8]:
#city와 state 필터링

df_geolocation['geolocation_city'] = df_geolocation['geolocation_city'].str.lower().apply(unidecode).str.strip().str.replace(r"\s+", " ", regex=True)

df_geolocation['geolocation_state'] = df_geolocation['geolocation_state'].str.upper().str.strip().str.replace(r"\s+", " ", regex=True)

In [9]:
df_geolocation.to_csv("../data/geolocation_clean.csv", index=False)